# Lab 3 — Build the Sample-Size Calculator and Run-Length Plan
# المختبر 3 — بناء حاسبة حجم العينة وخطة مدة التشغيل

**Module 3 · الوحدة 3 — Power Analysis and Sample Sizing / تحليل القوة الإحصائية وتحديد حجم العينة**
SDA-DSC-213 · Experimentation, A/B Testing and Causal Inference · SDAIA Academy

---

### The question every stakeholder asks first

*"How long do we have to run it?"*

An experimentation lead who cannot answer that credibly loses the room. Worse, an
under-powered experiment produces a stream of false negatives that teaches the
organisation that "experiments never find anything" — killing the experimentation
culture before it starts.

This lab turns the design doc you froze in Lab 2 into a **number of users** and a
**number of days**, and then buys some of those days back with variance reduction.

### What you will build

| # | Step | Minutes |
|---|---|---|
| 1 | `sample_size_proportions`, cross-checked against `statsmodels` | 8 |
| 2 | The **n-vs-MDE quadratic** — the plot you pin to the wall | 8 |
| 3 | The **MDE achievable at live Injaz traffic**, and the go / no-go call | 8 |
| 4 | The **run-length plan**: arms × trigger rate × the whole-week floor | 8 |
| 5 | **CUPED** variance reduction from the real pre-period file | 10 |
| 6 | The one-paragraph sizing recommendation for the design doc | 8 |

### The vocabulary this lab installs (bilingual, once)

- **statistical power (القوة الإحصائية)** — 1 − β, the probability of detecting a true effect
  *of a stated size*. "Is this experiment powered?" is an incomplete sentence.
- **Type I error / significance level α (خطأ النوع الأول / مستوى الدلالة)** — declaring an effect that is not there.
- **Type II error / β (خطأ النوع الثاني)** — missing an effect that is there.
- **MDE / minimum detectable effect (أصغر أثر قابل للكشف)** — the smallest effect this design
  can reliably find. A *business* choice, not a statistical one.
- **variance reduction / CUPED (خفض التباين)** — using pre-experiment data to shrink noise,
  which is arithmetically the same as buying extra sample for free.

> All code, variables and comments stay in **English**.


---
## Step 0 — Inputs

Two real files drive this lab:

- `injaz_pre_period.csv` — 50,000 users' pre-experiment behaviour. This is the CUPED
  covariate, and it is *pre-treatment* by construction, which is the whole safety argument.
- `injaz_traffic_forecast.csv` — 90 days of forecast eligible traffic and trigger rate.
  It follows the KSA working week: Friday and Saturday run at roughly 58% of a weekday.
  That single fact is why run lengths must be **whole weeks**.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `causal_utils` importable from anywhere under course_assets/
ROOT = Path.cwd()
while not (ROOT / "causal_utils").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from causal_utils import use_sdaia_style
use_sdaia_style()

RANDOM_SEED = 213          # fixed for every lab: reproducibility is graded
rng = np.random.default_rng(RANDOM_SEED)
print("Environment ready ·", ROOT)

In [ ]:
def needs(*values, msg="complete the TODO in the cell above, then re-run this cell"):
    """Guard for the start notebook: True (with a friendly note) if a TODO is unfinished."""
    if any(v is None for v in values):
        print(f"[not implemented yet] {msg}")
        return True
    return False

In [ ]:
import json
from scipy.stats import norm
from causal_utils import (sample_size_proportions, sample_size_means, power_for_n,
                          mde_for_n, run_length, cuped_adjust,
                          NAVY, BLUE, ORANGE, TEAL, SLATE, MUTED, PANEL)

with open(DATA / "GROUND_TRUTH.json") as f:
    GROUND_TRUTH = json.load(f)

pre = pd.read_csv(DATA / "injaz_pre_period.csv")
traffic = pd.read_csv(DATA / "injaz_traffic_forecast.csv", parse_dates=["date"])

BASELINE = 0.55          # Injaz platform completion rate, from the pre-period
TARGET_MDE = 0.02        # DESIGN_DOC["mde_absolute"] frozen in Lab 2
ALPHA, POWER = 0.05, 0.80

print(f"pre-period users : {len(pre):,}")
print(f"traffic days     : {len(traffic)}")
print()
print(traffic.head(8).to_string(index=False))
print()
weekday = traffic.loc[traffic["is_weekend"] == 0, "eligible_users"].mean()
weekend = traffic.loc[traffic["is_weekend"] == 1, "eligible_users"].mean()
print(f"mean eligible users, weekday : {weekday:,.0f}")
print(f"mean eligible users, weekend : {weekend:,.0f}   ({weekend / weekday:.0%} of a weekday)")
print(f"mean trigger rate            : {traffic['trigger_rate'].mean():.4f}")

---
## Step 1 *(8 min)* — Sample size for a difference in proportions

Fix any three of {α, power, MDE, n} and the fourth is determined. Solving for n:

$$n_{\text{per arm}} = \frac{(z_{1-\alpha/2} + z_{1-\beta})^2 \;\big[p_1(1-p_1) + p_2(1-p_2)\big]}{(p_2 - p_1)^2},
\qquad p_2 = p_1 + \text{MDE}$$

with z ≈ 1.96 for α = 0.05 two-sided and z ≈ 0.84 for 80% power.

Write the closed form yourself — it is the version that shows you the levers — then
cross-check against `statsmodels` (which works in Cohen's *h*) and against
`causal_utils.sample_size_proportions` (which uses the slightly more careful pooled-variance
form). All three must agree within rounding. **If two of them disagree, one is wrong and
finding out which is the exercise.**


In [ ]:
def my_sample_size_proportions(p1, mde, alpha=0.05, power=0.80):
    """TODO: per-arm sample size to detect an ABSOLUTE lift `mde` over baseline `p1`.

        p2 = p1 + mde
        z_a = norm.ppf(1 - alpha / 2)        # two-sided
        z_b = norm.ppf(power)
        n = (z_a + z_b)**2 * (p1*(1-p1) + p2*(1-p2)) / (p2 - p1)**2

    Round UP (np.ceil) and return an int. Raise ValueError if p2 leaves (0, 1).

    Self-check: at p1 = 0.55, MDE = 0.02, alpha = 0.05, power = 0.80 you should land
    within 1% of causal_utils.sample_size_proportions.
    """
    return None


from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

n_mine = my_sample_size_proportions(BASELINE, TARGET_MDE, ALPHA, POWER)
n_lib = sample_size_proportions(BASELINE, TARGET_MDE, alpha=ALPHA, power=POWER)
h = proportion_effectsize(BASELINE + TARGET_MDE, BASELINE)
n_sm = int(np.ceil(NormalIndPower().solve_power(effect_size=h, alpha=ALPHA,
                                                power=POWER, alternative="two-sided")))

if not needs(n_mine, msg="implement my_sample_size_proportions"):
    spread = max(n_mine, n_lib, n_sm) / min(n_mine, n_lib, n_sm) - 1
    assert spread < 0.01, (
        f"the three implementations disagree by {spread:.2%} — they must agree within 1%. "
        "Most likely you fed statsmodels an absolute lift instead of Cohen's h, "
        "or used a one-sided z.")
    print(f"  closed form (yours)          : {n_mine:>8,} per arm")
    print(f"  causal_utils (pooled form)   : {n_lib:>8,} per arm")
    print(f"  statsmodels (Cohen's h)      : {n_sm:>8,} per arm")
    print(f"\n  largest relative disagreement: {spread:.2%}   (benchmark: under 1%)")
    print(f"\nTotal users needed for a 2-arm test: {2 * n_lib:,}")

---
## Step 2 *(8 min)* — The quadratic that governs every conversation

n scales with **1 / MDE²**. Halving the effect you want to detect roughly **quadruples**
the sample. This is the single most important fact in the module: it explains why
detecting tiny effects is so expensive, and why picking an honest MDE matters more than
any other design choice.

Plot it once, keep it forever. Every future "can we detect something smaller?" gets
answered by pointing at this curve.


In [ ]:
# This step uses the LIBRARY sizing function so the curve is right even if your
# closed form is still a TODO. Swap in my_sample_size_proportions once it passes.
mde_grid = np.linspace(0.005, 0.05, 60)
n_grid = np.array([sample_size_proportions(BASELINE, m, alpha=ALPHA, power=POWER)
                   for m in mde_grid])

fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.plot(100 * mde_grid, n_grid / 1000, color=BLUE, lw=2.6)

for m, colour in [(0.02, TEAL), (0.01, ORANGE), (0.005, "#C0392B")]:
    n_here = sample_size_proportions(BASELINE, m, alpha=ALPHA, power=POWER)
    ax.plot([100 * m], [n_here / 1000], "o", color=colour, ms=10,
            markeredgecolor="white", markeredgewidth=1.6, zorder=5)
    ax.annotate(f"MDE {100*m:.1f} pp\n{n_here:,}/arm",
                xy=(100 * m, n_here / 1000), xytext=(100 * m + 0.35, n_here / 1000 + 12),
                fontsize=9.5, color=colour, fontweight="bold")

ax.set_xlabel("Minimum detectable effect (percentage points on completion rate)")
ax.set_ylabel("Required sample size per arm (thousands of users)")
ax.set_title(f"Required sample explodes as the MDE shrinks  (p1 = {BASELINE:.2f}, "
             f"alpha = {ALPHA}, power = {POWER:.0%})")
ax.set_xlim(0.3, 5.2)
ax.set_ylim(-8, 175)

n2 = sample_size_proportions(BASELINE, 0.02, alpha=ALPHA, power=POWER)
n1 = sample_size_proportions(BASELINE, 0.01, alpha=ALPHA, power=POWER)
ax.annotate(
    f"Takeaway: halving the MDE from 2.0 to 1.0 pp multiplies the sample by "
    f"{n1 / n2:.1f}x\n(from {n2:,} to {n1:,} per arm). n grows like 1 / MDE^2 — "
    "there is no way to buy\naround it except variance reduction (Step 5) or a bigger effect.",
    xy=(0.30, 0.62), xycoords="axes fraction", fontsize=10, color=SLATE,
    bbox=dict(boxstyle="round,pad=0.55", fc=PANEL, ec=MUTED, lw=0.8))
plt.tight_layout()
plt.show()

print(f"{'MDE (pp)':>10}{'n per arm':>14}{'x vs 2.0 pp':>14}")
print("-" * 38)
for m in [0.005, 0.0075, 0.01, 0.015, 0.02, 0.03, 0.05]:
    n_here = sample_size_proportions(BASELINE, m, alpha=ALPHA, power=POWER)
    print(f"{100*m:>10.2f}{n_here:>14,}{n_here / n2:>14.2f}")

---
## Step 3 *(8 min)* — Invert it: what can live traffic actually detect?

Now the honest pre-launch question. Given the traffic Injaz really has, what is the
smallest effect a four-week test could find at 80% power?

$$\text{MDE} = (z_{1-\alpha/2} + z_{1-\beta})\sqrt{\frac{2\,p_1(1-p_1)}{n_{\text{per arm}}}}$$

Two traffic scenarios, deliberately:

- **Planning assumption** — 12,000 eligible/day at a 0.60 trigger rate (the round numbers
  in the design review).
- **Forecast file** — the actual `injaz_traffic_forecast.csv` first 28 days, which is
  higher and includes the weekend dip.

Ask the question in the order that matters: *is a 2-point lift detectable?* — not
*was the result significant?*


In [ ]:
def my_mde_for_n(n_per_arm, p1, alpha=0.05, power=0.80):
    """TODO: the smallest absolute effect detectable with `n_per_arm` per arm.

        z_a = norm.ppf(1 - alpha / 2);  z_b = norm.ppf(power)
        se_unit = sqrt(2 * p1 * (1 - p1) / n_per_arm)
        MDE = (z_a + z_b) * se_unit

    Self-check: it must agree with causal_utils.mde_for_n to 4 decimal places.
    """
    return None


WINDOW_DAYS = 28
usable_a = 12_000 * 0.60 * WINDOW_DAYS
first28 = traffic.head(WINDOW_DAYS)
usable_b = float((first28["eligible_users"] * first28["trigger_rate"]).sum())

mde_a = my_mde_for_n(int(usable_a / 2), BASELINE, ALPHA, POWER)
mde_b = my_mde_for_n(int(usable_b / 2), BASELINE, ALPHA, POWER)

if not needs(mde_a, mde_b, msg="implement my_mde_for_n"):
    ref = mde_for_n(int(usable_b / 2), BASELINE, ALPHA, POWER)
    assert abs(mde_b - ref) < 1e-4, \
        f"your MDE {mde_b:.5f} disagrees with causal_utils {ref:.5f}"
    print(f"{'scenario':<34}{'triggered users':>18}{'per arm':>12}{'MDE':>10}")
    print("-" * 76)
    print(f"{'A. 12,000/day x 0.60 trigger':<34}{usable_a:>18,.0f}{usable_a/2:>12,.0f}"
          f"{100*mde_a:>9.3f}pp")
    print(f"{'B. forecast file, first 28 days':<34}{usable_b:>18,.0f}{usable_b/2:>12,.0f}"
          f"{100*mde_b:>9.3f}pp")
    verdict = "YES" if mde_b < TARGET_MDE else "NO"
    print(f"\nIs the frozen {100*TARGET_MDE:.1f} pp MDE detectable in four weeks? {verdict}")
    print("Write one sentence for the design doc explaining the answer.")

In [ ]:
def my_power_for_n(n_per_arm, p1, effect, alpha=0.05):
    """TODO: probability of detecting `effect` with `n_per_arm` per arm.

        p2 = p1 + effect
        se = sqrt(p1*(1-p1)/n + p2*(1-p2)/n)
        power = norm.cdf(abs(effect) / se - norm.ppf(1 - alpha/2))

    Self-check: must agree with causal_utils.power_for_n, and at the planned n it
    should come out close to 0.80 against the frozen 2 pp MDE.
    """
    return None


n_planned = sample_size_proportions(BASELINE, TARGET_MDE, alpha=ALPHA, power=POWER)
_probe = my_power_for_n(n_planned, BASELINE, TARGET_MDE, ALPHA)

if not needs(_probe, msg="implement my_power_for_n"):
    ref = power_for_n(n_planned, BASELINE, TARGET_MDE, alpha=ALPHA)
    assert abs(_probe - ref) < 1e-6, f"yours {_probe:.4f} vs causal_utils {ref:.4f}"
    assert 0.79 < _probe < 0.81, "at the planned n, power against the MDE should be ~0.80"

    effects = np.linspace(0.0, 0.045, 120)
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    for n_arm, colour, label in [
            (n_planned // 4, MUTED, f"n = {n_planned // 4:,}/arm (quarter size)"),
            (n_planned, BLUE, f"n = {n_planned:,}/arm (planned)"),
            (n_planned * 4, TEAL, f"n = {n_planned * 4:,}/arm (4x)")]:
        ax.plot(100 * effects, [100 * my_power_for_n(n_arm, BASELINE, e) for e in effects],
                color=colour, lw=2.4, label=label)
    ax.axhline(80, color=ORANGE, ls="--", lw=1.5, label="80% power convention")
    ax.axvline(100 * TARGET_MDE, color=NAVY, ls=":", lw=1.5,
               label=f"frozen MDE ({100*TARGET_MDE:.1f} pp)")
    ax.set_xlabel("True effect on completion rate (percentage points)")
    ax.set_ylabel("Probability of detecting it (%)")
    ax.set_title("Power is a property of a design against a specific effect size")
    ax.legend(loc="lower right")
    ax.set_ylim(0, 103)
    ax.annotate(
        f"TODO: write the takeaway. What is the power against a 1.0 pp effect?\n"
        f"(hint: my_power_for_n(n_planned, BASELINE, 0.01) = "
        f"{100 * my_power_for_n(n_planned, BASELINE, 0.01):.0f}%)",
        xy=(0.03, 0.62), xycoords="axes fraction", fontsize=10, color=SLATE,
        bbox=dict(boxstyle="round,pad=0.55", fc=PANEL, ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()

### ✅ Checkpoint A — what you should be seeing by now

≈24 minutes in. A sizing function agreeing with two independent implementations, the
quadratic curve, and the MDE your live traffic can actually resolve.


In [ ]:
def checkpoint_a():
    if n_mine is None or mde_b is None:
        print("✗ Steps 1-3 incomplete.")
        return
    print(f"✓ n per arm (MDE {100*TARGET_MDE:.1f} pp)  : {n_mine:,}   "
          f"(causal_utils {n_lib:,}, statsmodels {n_sm:,})")
    print(f"✓ quadratic confirmed          : MDE 1.0 pp needs "
          f"{sample_size_proportions(BASELINE, 0.01):,}/arm "
          f"= {sample_size_proportions(BASELINE, 0.01) / n_lib:.1f}x")
    print(f"✓ MDE at 4 weeks of traffic    : {100*mde_b:.3f} pp")
    print(f"✓ is the frozen MDE reachable  : {'yes' if mde_b < TARGET_MDE else 'NO'}")
    print()
    print("Next: turn the sample size into a calendar, then buy days back with CUPED.")

checkpoint_a()

---
## Step 4 *(8 min)* — From sample size to a calendar

Three things bite teams here, all of them avoidable:

1. **n is *per arm*.** A 3-arm test needs three times the per-arm figure. Teams routinely
   forget the multiplier and under-run.
2. **Only *triggered* users count.** If 38% of visitors never reach the upload step, your
   effective traffic is 62% of the headline number.
3. **A whole number of weeks is a floor.** Weekday and weekend users are different people.
   Stopping on a Thursday because the arithmetic said 5.2 days samples a biased slice of
   the population — and novelty effects in the first days argue for the same floor.

**The floor this lab uses is `floor_days=14` — two whole weeks.** One week already
guarantees that Friday and Saturday users (about 58% of a weekday's traffic, and a
different population) are sampled in proportion; the *second* week is what lets you
see whether the first week's numbers were a novelty spike, and it gives you a
week-on-week comparison rather than a single draw. Every `run_length(...)` call
below therefore passes `floor_days=14`, and at the frozen 2.0 pp MDE that floor —
not the arithmetic — is what sets the calendar at **14 days**.

Build the planner, then compare with `causal_utils.run_length`.


In [ ]:
def my_run_length(n_per_arm, daily_eligible, trigger_rate, n_arms=2, floor_days=14):
    """TODO: turn a per-arm sample size into a calendar commitment.

      total          = n_per_arm * n_arms
      daily_usable   = daily_eligible * trigger_rate
      days_arithmetic= ceil(total / daily_usable)
      recommended    = max(days_arithmetic, floor_days), then rounded UP to a
                       whole number of weeks

    Return a dict with keys: total_units_needed, daily_usable,
    days_by_arithmetic, days_recommended.

    Self-check: must agree with causal_utils.run_length(..., floor_days=14).
    """
    return None


daily_eligible = float(traffic["eligible_users"].mean())
trigger_rate = float(traffic["trigger_rate"].mean())
_probe = my_run_length(sample_size_proportions(BASELINE, TARGET_MDE),
                       daily_eligible, trigger_rate, n_arms=2, floor_days=14)

if not needs(_probe, msg="implement my_run_length"):
    ref = run_length(sample_size_proportions(BASELINE, TARGET_MDE), daily_eligible,
                     trigger_rate, n_arms=2, floor_days=14)
    assert _probe["days_recommended"] == ref.days_recommended, \
        f"yours {_probe['days_recommended']} vs causal_utils {ref.days_recommended}"
    assert _probe["days_recommended"] % 7 == 0, "the plan must be a whole number of weeks"

    rows = []
    for mde in [0.005, 0.0075, 0.01, 0.015, 0.02, 0.03]:
        n_arm = sample_size_proportions(BASELINE, mde, alpha=ALPHA, power=POWER)
        for arms in (2, 3):
            p = my_run_length(n_arm, daily_eligible, trigger_rate,
                              n_arms=arms, floor_days=14)
            rows.append({"MDE_pp": round(100 * mde, 2), "arms": arms, "n_per_arm": n_arm,
                         "total_needed": p["total_units_needed"],
                         "days_arithmetic": p["days_by_arithmetic"],
                         "days_recommended": p["days_recommended"]})
    plan_table = pd.DataFrame(rows)
    print(plan_table.to_string(index=False))
    print("\ncausal_utils cross-check for the frozen 2.0 pp / 2-arm design:")
    print(ref)
else:
    plan_table = None

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
first28 = traffic.head(28)
usable = first28["eligible_users"] * first28["trigger_rate"]
colours = [ORANGE if w else BLUE for w in first28["is_weekend"]]
ax.bar(range(len(first28)), usable / 1000, color=colours, width=0.72)
ax.set_xticks(range(0, len(first28), 2))
ax.set_xticklabels([d.strftime("%a %d") for d in first28["date"][::2]], rotation=45,
                   ha="right", fontsize=8)
ax.set_ylabel("Triggered users available (thousands/day)")
ax.set_xlabel("Forecast day (blue = weekday, orange = Fri/Sat)")
ax.set_title("Why a run length must be a whole number of weeks")
wk = usable[first28["is_weekend"] == 0].mean()
we = usable[first28["is_weekend"] == 1].mean()
ax.axhline(wk / 1000, color=BLUE, ls="--", lw=1.3)
ax.axhline(we / 1000, color=ORANGE, ls="--", lw=1.3)
ax.annotate(
    f"Takeaway: Fri/Sat deliver {we:,.0f} usable users against {wk:,.0f} on a weekday "
    f"({we/wk:.0%}).\nStopping mid-week does not just shorten the test, it changes WHO is in it.",
    xy=(0.02, 0.06), xycoords="axes fraction", fontsize=10, color=SLATE,
    bbox=dict(boxstyle="round,pad=0.55", fc=PANEL, ec=MUTED, lw=0.8))
plt.tight_layout()
plt.show()

---
## Step 5 *(10 min)* — CUPED: buy sample size for free

Because n scales with variance, **reducing variance is arithmetically identical to buying
sample**. The headline technique is **CUPED** — Controlled-experiment Using Pre-Experiment
Data:

$$Y^{\text{cuped}} = Y - \theta\,(X_{\text{pre}} - \bar{X}_{\text{pre}}), \qquad \theta = \frac{\text{Cov}(Y, X_{\text{pre}})}{\text{Var}(X_{\text{pre}})}$$

The variance falls by ρ², where ρ = corr(Y, X_pre). The entire safety argument is one
condition: **X_pre is measured before assignment**, so treatment cannot have affected it,
so the adjustment cannot bias the effect. Never CUPED on a post-treatment covariate.

We use the real `injaz_pre_period.csv` joined to the experiment outcomes, restricted to
triggered users — the population the frozen plan actually analyses.


In [ ]:
def my_cuped_adjust(y, x_pre):
    """TODO: CUPED-adjust an outcome using a pre-experiment covariate.

        theta = cov(y, x) / var(x)                    (ddof=1 throughout)
        y_adj = y - theta * (x - mean(x))
        variance_reduction = 1 - var(y_adj) / var(y)
        rho = corr(y, x)

    Drop rows where either value is NaN before computing anything.
    Return (y_adj, theta, variance_reduction, rho).

    Self-check: on this data the reduction is ~14% and must equal rho^2 to
    within rounding. It must also match causal_utils.cuped_adjust.
    """
    return None


results = pd.read_csv(DATA / "injaz_experiment_results.csv")
joined = results.merge(pre, on="user_id", how="inner")
triggered = joined.loc[joined["triggered"] == 1]

_out = my_cuped_adjust(triggered["completed"], triggered["pre_completion_rate"])

if not needs(_out, msg="implement my_cuped_adjust"):
    y_adj, theta, var_reduction, rho = _out
    ref = cuped_adjust(triggered["completed"], triggered["pre_completion_rate"])
    assert abs(var_reduction - ref.variance_reduction) < 1e-6, \
        f"yours {var_reduction:.4f} vs causal_utils {ref.variance_reduction:.4f}"
    assert abs(var_reduction - rho**2) < 0.005, \
        "the variance reduction must equal rho^2 — check your theta"
    assert 0.10 < var_reduction < 0.20, \
        f"expected ~14% on this covariate, got {var_reduction:.1%}"
    print(f"corr(pre_completion_rate, completed): {rho:.4f}")
    print(f"theta                               : {theta:.4f}")
    print(f"variance reduction                  : {var_reduction:.1%}")
    print(f"rho^2 (the theoretical bound)       : {rho**2:.1%}")
    print(f"effective sample multiplier         : {1 / (1 - var_reduction):.3f}x")
    print("\ncausal_utils cross-check:")
    print(ref)
else:
    var_reduction = None
    rho = None

In [ ]:
if needs(var_reduction, msg="finish Step 5 to draw this"):
    pass
else:
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

    ax = axes[0]
    sub = triggered.sample(4_000, random_state=RANDOM_SEED)
    jitter = np.random.default_rng(RANDOM_SEED).normal(0, 0.03, len(sub))
    ax.scatter(sub["pre_completion_rate"], sub["completed"] + jitter, s=6, alpha=0.18,
               color=BLUE, edgecolors="none")
    bins = pd.cut(triggered["pre_completion_rate"], 12)
    binned = triggered.groupby(bins, observed=True).agg(
        x=("pre_completion_rate", "mean"), y=("completed", "mean")).dropna()
    ax.plot(binned["x"], binned["y"], "o-", color=ORANGE, lw=2.4, ms=7,
            label="binned mean completion")
    ax.set_xlabel("Pre-period completion rate (measured BEFORE assignment)")
    ax.set_ylabel("Completed in the experiment (0/1, jittered)")
    ax.set_title(f"The covariate is predictive: rho = {rho:.3f}")
    ax.legend(loc="upper left")

    ax = axes[1]
    raw = triggered["completed"].to_numpy(dtype=float)
    ax.hist(raw - raw.mean(), bins=40, alpha=0.55, color=MUTED,
            label=f"raw outcome  (var {np.var(raw, ddof=1):.4f})")
    ax.hist(y_adj - y_adj.mean(), bins=40, alpha=0.75, color=TEAL,
            label=f"CUPED-adjusted  (var {np.var(y_adj, ddof=1):.4f})")
    ax.set_xlabel("Centred outcome (completion, in units of the rate)")
    ax.set_ylabel("Users")
    ax.set_title(f"Variance falls {var_reduction:.1%} — worth {1/(1-var_reduction):.2f}x the sample")
    ax.legend(loc="upper right", fontsize=9)

    fig.suptitle("CUPED turns a pre-experiment covariate into free sample size",
                 fontsize=13, fontweight="bold", color=NAVY)
    fig.text(0.5, -0.05,
             f"Takeaway: a covariate correlated {rho:.2f} with the outcome removes {var_reduction:.0%} "
             f"of the variance,\nwhich is arithmetically the same as running the experiment on "
             f"{1/(1-var_reduction):.2f}x as many users — for zero extra days of traffic.",
             ha="center", fontsize=10, color=SLATE,
             bbox=dict(boxstyle="round,pad=0.55", fc=PANEL, ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()

In [ ]:
if needs(var_reduction, plan_table, msg="finish Steps 4-5"):
    pass
else:
    n_plain = sample_size_proportions(BASELINE, TARGET_MDE, alpha=ALPHA, power=POWER)
    n_cuped = int(np.ceil(n_plain * (1 - var_reduction)))

    plan_plain = my_run_length(n_plain, daily_eligible, trigger_rate, n_arms=2, floor_days=14)
    plan_cuped = my_run_length(n_cuped, daily_eligible, trigger_rate, n_arms=2, floor_days=14)

    print(f"{'design':<26}{'n/arm':>10}{'total':>12}{'days (arith.)':>16}{'days (plan)':>14}")
    print("-" * 80)
    print(f"{'no variance reduction':<26}{n_plain:>10,}{2*n_plain:>12,}"
          f"{plan_plain['days_by_arithmetic']:>16}{plan_plain['days_recommended']:>14}")
    print(f"{'CUPED-adjusted':<26}{n_cuped:>10,}{2*n_cuped:>12,}"
          f"{plan_cuped['days_by_arithmetic']:>16}{plan_cuped['days_recommended']:>14}")
    print()
    print(f"CUPED removes {n_plain - n_cuped:,} users per arm ({var_reduction:.1%}).")
    print("At this MDE the 14-day seasonality floor already dominates, so the calendar does")
    print("not move — the saving shows up instead as a TIGHTER confidence interval at the same")
    print("run length, which is what Lab 4 will actually report.")
    print()
    print("Where CUPED does change the calendar — a smaller MDE, where arithmetic dominates:")
    for mde in [0.005, 0.0075, 0.01]:
        a = sample_size_proportions(BASELINE, mde, alpha=ALPHA, power=POWER)
        b = int(np.ceil(a * (1 - var_reduction)))
        da = my_run_length(a, daily_eligible, trigger_rate, 2, 14)["days_recommended"]
        db = my_run_length(b, daily_eligible, trigger_rate, 2, 14)["days_recommended"]
        print(f"  MDE {100*mde:>4.2f} pp : {a:>7,}/arm -> {b:>7,}/arm    "
              f"{da:>3} days -> {db:>3} days")

### ✅ Checkpoint B — what you should be seeing by now

≈42 minutes in. A run-length table where the arm multiplier is visible, and a CUPED
variance reduction around 13.5% that matches ρ².


In [ ]:
def checkpoint_b():
    if plan_table is None or var_reduction is None:
        print("✗ Steps 4-5 incomplete.")
        return
    frozen = plan_table.query("MDE_pp == 2.0 and arms == 2").iloc[0]
    three_arm = plan_table.query("MDE_pp == 2.0 and arms == 3").iloc[0]
    print(f"✓ frozen design (2.0 pp, 2 arms) : {int(frozen['n_per_arm']):,}/arm, "
          f"{int(frozen['total_needed']):,} users, run {int(frozen['days_recommended'])} days")
    print(f"✓ arm multiplier visible         : 3 arms need "
          f"{int(three_arm['total_needed']):,} users")
    print(f"✓ CUPED variance reduction       : {var_reduction:.1%}  (rho^2 = {rho**2:.1%})")
    print(f"✓ effective sample multiplier    : {1 / (1 - var_reduction):.2f}x")
    print()
    print("Next: grade against ground truth, then write the sizing paragraph.")

checkpoint_b()

---
## Step 6 — Grade yourself against the planted ground truth

`GROUND_TRUTH.json` records what the generator built into `injaz_pre_period.csv`. Your
recovered correlation and variance reduction should sit on top of it.


In [ ]:
if needs(var_reduction, rho, n_mine, msg="finish Steps 1-5 first"):
    pass
else:
    M3 = GROUND_TRUTH["module3_pre_period"]
    M3T = GROUND_TRUTH["module3_traffic"]

    print(f"{'quantity':<44}{'recovered':>12}{'planted':>12}   verdict")
    print("-" * 82)

    rows = [
        ("corr(pre_completion_rate, completed)", rho, M3["corr_pre_vs_outcome"], 0.02),
        ("CUPED variance reduction (%)", 100 * var_reduction,
         M3["expected_cuped_variance_reduction_pct"], 2.0),
        ("mean daily eligible users", daily_eligible, M3T["mean_daily_eligible"], 50),
        ("mean daily triggered users", daily_eligible * trigger_rate,
         M3T["mean_daily_triggered"], 50),
    ]
    for name, got, want, tol in rows:
        print(f"{name:<44}{got:>12.4f}{want:>12.4f}   "
              f"{'MATCH' if abs(got - want) <= tol else 'CHECK'}")

    print()
    print("Benchmarks for Module 3:")
    print(f"  closed-form vs statsmodels n agree within 1%   : "
          f"{abs(n_mine / n_sm - 1) < 0.01}   ({abs(n_mine / n_sm - 1):.3%})")
    print(f"  CUPED estimate within 2% of the planted value  : "
          f"{abs(100 * var_reduction - M3['expected_cuped_variance_reduction_pct']) < 2}")
    print(f"  run-length plan includes arms x trigger x floor: True")

---
## Step 7 *(8 min)* — The sizing paragraph for the design doc

One paragraph. It must contain: the chosen MDE and why, n per arm, number of arms, the
run length with and without CUPED, and an explicit feasibility verdict. No formulas.


In [ ]:
SIZING_RECOMMENDATION = """
TODO - one paragraph (or four short ones). It must state:
  1. the chosen MDE and WHY that number (a business argument, not a statistical one)
  2. n per arm and the number of arms
  3. the run length, and why the whole-week floor overrides the arithmetic
  4. the CUPED-adjusted numbers and what the saving actually buys
  5. an explicit GO / NO-GO feasibility verdict, including what you are NOT powered to say
"""
print(SIZING_RECOMMENDATION)

---
## What you now own

In [ ]:
print("""
TOOLKIT CONTRIBUTION — Lab 3
============================================================
Reusable artefacts you built and can lift into any project:

  my_sample_size_proportions(p1, mde, alpha, power)
      Per-arm sizing for a rate metric, cross-validated against statsmodels
      and causal_utils. The closed form is the one you argue from.

  my_mde_for_n(n_per_arm, p1, alpha, power)
      The pre-launch feasibility question, in one line. Run it BEFORE the
      experiment so 'no effect' can never be confused with 'no detectable
      effect here'.

  my_power_for_n(n_per_arm, p1, effect, alpha)
      Power against a SPECIFIC effect size, plus the power-curve chart.

  my_run_length(n_per_arm, daily_eligible, trigger_rate, n_arms, floor_days)
      Sample size -> calendar, with the three things teams forget: the arm
      multiplier, the trigger rate, and the whole-week floor.

  my_cuped_adjust(y, x_pre)
      Variance reduction from pre-experiment data. Unbiased ONLY because the
      covariate is pre-treatment - never CUPED on a post-treatment column.

  The n-vs-MDE quadratic plot
      Pin it to the wall. Every 'can we detect something smaller?' gets
      answered by pointing at it.

Carried forward:
  - Lab 4 analyses at exactly this n, once, at the frozen horizon - and uses
    the same pre-period covariate to tighten the interval.
  - The capstone sizes the autofill experiment with these functions against a
    1.0 pp practical threshold.
============================================================
""")

---
## Mini-exercises

### 1 · Quiz

1. Define statistical power.
2. If you halve the MDE, how does the required n change?
3. Why is a non-significant result not proof of no effect?
4. Why must you multiply the per-arm n by the number of arms?
5. Why is CUPED's covariate required to be pre-treatment?

### 2 · Debugging exercise — the missing arm multiplier

The cell below sizes a **four-arm** test but reports the two-arm total. Find the under-run:
how many users short is the plan, and how many days short is the calendar?

### 3 · Estimation exercise

Baseline 0.30, required MDE 0.03, **90%** power. Estimate n by hand from the closed form
first (z₀.₉₇₅ ≈ 1.96, z₀.₉₀ ≈ 1.28), then verify in code. How much more does 90% power
cost than 80%?

### 4 · Discussion

- Leadership wants to detect a 0.3-point lift on this traffic in two weeks. Walk them
  through why that is infeasible, and name the three levers available.
- When is 90% power worth the extra sample over 80%?


In [ ]:
if needs(plan_table, msg="finish Step 4 to run the mini-exercises"):
    pass
else:
    # --- Mini-exercise 2: the missing arm multiplier --------------------------
    n_arm_4 = sample_size_proportions(BASELINE, TARGET_MDE, alpha=ALPHA, power=POWER)

    wrong = my_run_length(n_arm_4, daily_eligible, trigger_rate, n_arms=2, floor_days=14)
    right = my_run_length(n_arm_4, daily_eligible, trigger_rate, n_arms=4, floor_days=14)

    print("A four-arm test (control + three treatments), sized at "
          f"{n_arm_4:,} per arm\n")
    print(f"{'':<26}{'total users':>14}{'days (arith.)':>16}{'days (plan)':>14}")
    print("-" * 70)
    print(f"{'reported (n_arms=2)':<26}{wrong['total_units_needed']:>14,}"
          f"{wrong['days_by_arithmetic']:>16}{wrong['days_recommended']:>14}")
    print(f"{'correct  (n_arms=4)':<26}{right['total_units_needed']:>14,}"
          f"{right['days_by_arithmetic']:>16}{right['days_recommended']:>14}")
    print()
    print(f"Under-run: {right['total_units_needed'] - wrong['total_units_needed']:,} users "
          f"and {right['days_by_arithmetic'] - wrong['days_by_arithmetic']} days of arithmetic.")
    print("Every arm is then under-powered, and the team reads a string of nulls as evidence")
    print("that none of the three treatments works.")

    # --- Mini-exercise 3: 80% vs 90% power -----------------------------------
    print("\n--- 90% power costs how much more? ---")
    for pw in (0.80, 0.90):
        n_here = sample_size_proportions(0.30, 0.03, alpha=0.05, power=pw)
        print(f"  p1=0.30, MDE=0.03, power={pw:.0%} : {n_here:,} per arm")
    n80 = sample_size_proportions(0.30, 0.03, alpha=0.05, power=0.80)
    n90 = sample_size_proportions(0.30, 0.03, alpha=0.05, power=0.90)
    print(f"  going from 80% to 90% power costs {n90 / n80 - 1:.0%} more sample")